Structured output:
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


Pydantic:
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

# Pydantic

In [1]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from IPython.display import display, Markdown

model = ChatNVIDIA(
  base_url="https://integrate.api.nvidia.com/v1",
  model="nvidia/nemotron-3.5-super-vl-120b-a12b",
  api_key=os.environ["NVIDIA_API_KEY"],
  temperature=1,
  top_p=0.95,
  max_completion_tokens=16384,
  seed=42,
)

try:
  response= model.invoke("My name is rudra , So tell me what is my name")
  print(display(response.content))
except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

'Your name is Rudra.'

None


In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The moview rating out of 10")
    

In [3]:
model_with_structure = model.with_structured_output(Movie)
response  = model_with_structure.invoke("Provide the details about the movie Suryavansham")
response

Movie(title='Suryavansham', year=1999, director='E.V.V. Satyanarayana', rating=5.2)

### Message out alongside parsed structure

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The moview rating out of 10")
    
model_with_structure= model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about the movie suryavansham")
response

Movie(title='Suryavansham', year=1999, director='E.V.V. Satyanarayana', rating=300000000.0)

In [5]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str=Field(description="Name of the Actor")
    role:str=Field(description="Role of the actor")

class MovieDetails(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The moview rating out of 10")
    cast :list[Actor]=Field(description="All the actor of the movie")
    genre:list[str]

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("please provide the detail of the movie Uri")

response
    

MovieDetails(title='Uri: The Surgical Strike (2019)', year=2019, director='Aditya Dhar', rating=7.9, cast=[Actor(name='Vicky Kaushal', role='Major Vihaan Singh Shergill'), Actor(name='Yami Gautam', role='Pallavi Sharma'), Actor(name='Paresh Rawal', role='National Security Advisor Govind Bhardwaj'), Actor(name='Mohit Raina', role='Major Karan Kashyap'), Actor(name='Kirti Kulhari', role='Commander Subha Sengupta'), Actor(name='Swaroop Sampat', role="Vihaan's Mother")], genre=['Action', 'War', 'Drama', 'Thriller'])

# TypeDict

In [6]:
from typing_extensions import TypedDict,Annotated


class Actor(BaseModel):
    name:Annotated[str,...,"The name of the actor"]
    role:Annotated[str,...,"The role of the actor"]

class MovieDetails(BaseModel):
    """A movie with details"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year of the movie"]
    director:Annotated[str,...,"The director name of the movie"]
    rating:Annotated[float,...,"The rating of the movie"]
    cast :Annotated[list[Actor],...,"The cast of the movie"]
    genre:Annotated[list[str],...,"The genre of the movie"]

model_with_typedict = model.with_structured_output(MovieDetails)
response = model_with_typedict.invoke("please provide the detail of the movie Uri")

response

MovieDetails(title='Uri: The Surgical Strike', year=2019, director='Aditya Dhar', rating=7.8, cast=[Actor(name='Vicky Kaushal', role='Major Vihaan Singh Shergill'), Actor(name='Yami Gautam', role='Pallavi Sharma'), Actor(name='Mohit Raina', role='Major Karan Kashyap'), Actor(name='Paresh Rawal', role='National Security Advisor Govind Bhardwaj'), Actor(name='Kirti Kulhari', role='Major Seerat Kaur'), Actor(name='Rajit Kapur', role='Prime Minister'), Actor(name='Swaroop Sampat', role="Vihaan's mother")], genre=['Action', 'War', 'Drama', 'Thriller'])

In [7]:
model.profile

{'name': 'Nemotron 3.5 Super VL',
 'text_inputs': True,
 'image_inputs': True,
 'text_outputs': True,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True}

# Data classes

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

# Comparision of pydantic vs TypeDict vs Dataclasses

In [8]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='f2e2eb4a-2c03-4c4e-bd29-c16cedd19522'),
  AIMessage(content='{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567"\n\nI should identify:\n- Name: John Doe\n- Email: john@example.com\n- Phone: (555) 123-4567\n\nI should present this clearly, perhaps in a structured format like bullet points or a list.\n\nNo complex reasoning needed here. Just straightforward extraction.', '_reasoning_api_fields': ['reasoning_content']}, response_metadata={'content': '{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}', 'role': 'assistant', 'reasoning_content': 'The user wants me to extract contact information from the given string: "John Doe, jo

In [9]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [10]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [11]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')